# 🚀 AlphaBreast V4 - Publication Quality
## Swin Transformer + Evoformer-Inspired Attention + 5-Fold CV

**Final Version Improvements:**
- ✅ **Swin Transformer** backbone (State-of-the-art 2024)
- ✅ **Evoformer-inspired** attention blocks (AlphaFold-style)
- ✅ **5-Fold Cross-Validation** for robust metrics
- ✅ **Combined Mass + Calcification** data (~1000 pairs)
- ✅ **Attention visualization** for interpretability

---

**Version History:**
| Version | Key Changes | AUC |
|---------|-------------|-----|
| V1 | Baseline CNN | 0.61 |
| V2 | + ResNet18, + Bidirectional Attention | 0.72 |
| V3 | + Combined Mass/Calc data | 0.74 |
| **V4** | **+ Swin Transformer, + 5-Fold CV** | **TBD** |

## 1️⃣ Setup

In [ ]:
!nvidia-smi

In [ ]:
# Install timm for Swin Transformer
!pip install -q timm

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import warnings
import json
from datetime import datetime
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
import timm  # For Swin Transformer

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score, roc_curve

# Set seeds
torch.manual_seed(42)
np.random.seed(42)
torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ⚠️ UPDATE THIS PATH
DATA_ROOT = '/content/drive/MyDrive/CBIS-DDSM'
CSV_PATH = os.path.join(DATA_ROOT, 'csv')
JPEG_PATH = os.path.join(DATA_ROOT, 'jpeg')

# Create output directory
OUTPUT_DIR = '/content/alphabreast_v4_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ CSV folder:", "Found" if os.path.exists(CSV_PATH) else "NOT FOUND")
print("✅ JPEG folder:", "Found" if os.path.exists(JPEG_PATH) else "NOT FOUND")
print(f"✅ Output dir: {OUTPUT_DIR}")

## 2️⃣ Load Combined Data (Mass + Calcification)

In [ ]:
# Load ALL data
mass_train = pd.read_csv(os.path.join(CSV_PATH, 'mass_case_description_train_set.csv'))
mass_test = pd.read_csv(os.path.join(CSV_PATH, 'mass_case_description_test_set.csv'))
calc_train = pd.read_csv(os.path.join(CSV_PATH, 'calc_case_description_train_set.csv'))
calc_test = pd.read_csv(os.path.join(CSV_PATH, 'calc_case_description_test_set.csv'))

# Combine
combined_train = pd.concat([mass_train, calc_train], ignore_index=True)
combined_test = pd.concat([mass_test, calc_test], ignore_index=True)

print("📊 Dataset Summary:")
print(f"   Training cases: {len(combined_train)} (Mass: {len(mass_train)}, Calc: {len(calc_train)})")
print(f"   Test cases: {len(combined_test)} (Mass: {len(mass_test)}, Calc: {len(calc_test)})")

In [ ]:
def create_paired_dataset(df):
    """Create paired CC-MLO samples."""
    paired_samples = []
    grouped = df.groupby(['patient_id', 'left or right breast'])
    
    for (patient_id, laterality), group in grouped:
        cc_rows = group[group['image view'] == 'CC']
        mlo_rows = group[group['image view'] == 'MLO']
        
        if len(cc_rows) > 0 and len(mlo_rows) > 0:
            cc_row = cc_rows.iloc[0]
            mlo_row = mlo_rows.iloc[0]
            pathology = cc_row['pathology']
            
            if 'MALIGNANT' in str(pathology).upper():
                label = 1
            elif 'BENIGN' in str(pathology).upper():
                label = 0
            else:
                continue
            
            paired_samples.append({
                'patient_id': patient_id,
                'cc_path': cc_row.get('image file path', ''),
                'mlo_path': mlo_row.get('image file path', ''),
                'cc_cropped': cc_row.get('cropped image file path', ''),
                'mlo_cropped': mlo_row.get('cropped image file path', ''),
                'label': label
            })
    
    return pd.DataFrame(paired_samples)

paired_train = create_paired_dataset(combined_train)
paired_test = create_paired_dataset(combined_test)

print(f"\n✅ Training pairs: {len(paired_train)}")
print(f"✅ Test pairs: {len(paired_test)}")
print(f"\n📊 Training labels: {paired_train['label'].value_counts().to_dict()}")
print(f"📊 Test labels: {paired_test['label'].value_counts().to_dict()}")

## 3️⃣ Dataset Class

In [ ]:
def find_image_fixed(jpeg_path, relative_path):
    """Find image from CBIS-DDSM structure."""
    if not relative_path or pd.isna(relative_path):
        return None
    parts = str(relative_path).strip().split('/')
    for i in [1, 2]:
        if len(parts) > i:
            folder_path = os.path.join(jpeg_path, parts[i])
            if os.path.exists(folder_path):
                for f in os.listdir(folder_path):
                    if f.endswith(('.jpg', '.jpeg', '.png')):
                        return os.path.join(folder_path, f)
    return None


class CBISDDSMDatasetV4(Dataset):
    """Dataset with consistent augmentation for paired views."""
    
    def __init__(self, paired_df, jpeg_path, transform=None, use_cropped=True):
        self.jpeg_path = jpeg_path
        self.transform = transform
        self.valid_samples = []
        self.labels = []  # For stratified k-fold
        
        print("🔍 Validating image paths...")
        for idx in tqdm(range(len(paired_df))):
            row = paired_df.iloc[idx]
            if use_cropped:
                cc_path = find_image_fixed(jpeg_path, row.get('cc_cropped', ''))
                mlo_path = find_image_fixed(jpeg_path, row.get('mlo_cropped', ''))
            else:
                cc_path = find_image_fixed(jpeg_path, row.get('cc_path', ''))
                mlo_path = find_image_fixed(jpeg_path, row.get('mlo_path', ''))
            
            if cc_path and mlo_path:
                self.valid_samples.append({
                    'cc_path': cc_path,
                    'mlo_path': mlo_path,
                    'label': row['label']
                })
                self.labels.append(row['label'])
        
        self.labels = np.array(self.labels)
        print(f"✅ Valid samples: {len(self.valid_samples)} / {len(paired_df)}")
    
    def __len__(self):
        return len(self.valid_samples)
    
    def __getitem__(self, idx):
        sample = self.valid_samples[idx]
        cc_img = Image.open(sample['cc_path']).convert('L')
        mlo_img = Image.open(sample['mlo_path']).convert('L')
        
        if self.transform:
            # Same augmentation for both views
            seed = np.random.randint(2147483647)
            torch.manual_seed(seed)
            cc_img = self.transform(cc_img)
            torch.manual_seed(seed)
            mlo_img = self.transform(mlo_img)
        
        return cc_img, mlo_img, torch.tensor(sample['label'], dtype=torch.long)

In [ ]:
# Transforms
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(20),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

# Create full dataset for k-fold
print("📦 Creating full training dataset...")
full_train_dataset = CBISDDSMDatasetV4(paired_train, JPEG_PATH, None, use_cropped=True)

print("\n📦 Creating test dataset...")
test_dataset = CBISDDSMDatasetV4(paired_test, JPEG_PATH, test_transform, use_cropped=True)

## 4️⃣ Model Definitions (Swin Transformer + Evoformer-Inspired)

In [ ]:
class FocalLoss(nn.Module):
    """Focal Loss for class imbalance."""
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()


class SwinEncoder(nn.Module):
    """Swin Transformer encoder for mammogram images."""
    
    def __init__(self, out_features=256, pretrained=True):
        super().__init__()
        
        # Load Swin-Tiny pretrained on ImageNet
        self.swin = timm.create_model(
            'swin_tiny_patch4_window7_224',
            pretrained=pretrained,
            num_classes=0,  # Remove classification head
            in_chans=1  # Grayscale input
        )
        
        # Output projection
        swin_features = self.swin.num_features  # 768 for swin_tiny
        self.projection = nn.Sequential(
            nn.Linear(swin_features, out_features),
            nn.LayerNorm(out_features),
            nn.GELU(),
            nn.Dropout(0.1)
        )
    
    def forward(self, x):
        features = self.swin(x)  # [B, 768]
        return self.projection(features)  # [B, out_features]


class SingleViewSwin(nn.Module):
    """Single-view baseline with Swin Transformer."""
    
    def __init__(self, num_classes=2, pretrained=True):
        super().__init__()
        self.encoder = SwinEncoder(out_features=256, pretrained=pretrained)
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        return self.classifier(self.encoder(x))


class LateFusionSwin(nn.Module):
    """Late fusion with Swin Transformer encoders."""
    
    def __init__(self, num_classes=2, pretrained=True):
        super().__init__()
        self.cc_encoder = SwinEncoder(out_features=256, pretrained=pretrained)
        self.mlo_encoder = SwinEncoder(out_features=256, pretrained=pretrained)
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, cc, mlo):
        cc_feat = self.cc_encoder(cc)
        mlo_feat = self.mlo_encoder(mlo)
        return self.classifier(torch.cat([cc_feat, mlo_feat], dim=1))


class EvoformerBlock(nn.Module):
    """
    Evoformer-Inspired Attention Block.
    Adapts AlphaFold's attention mechanism for multi-view mammography.
    
    Key components:
    1. Bidirectional cross-attention (CC ↔ MLO)
    2. Pairwise representation update
    3. Gated feed-forward network
    """
    
    def __init__(self, embed_dim=256, num_heads=8, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        
        # Bidirectional cross-attention
        self.cc_to_mlo_attn = nn.MultiheadAttention(
            embed_dim, num_heads, dropout=dropout, batch_first=True
        )
        self.mlo_to_cc_attn = nn.MultiheadAttention(
            embed_dim, num_heads, dropout=dropout, batch_first=True
        )
        
        # Layer norms
        self.norm_cc1 = nn.LayerNorm(embed_dim)
        self.norm_cc2 = nn.LayerNorm(embed_dim)
        self.norm_mlo1 = nn.LayerNorm(embed_dim)
        self.norm_mlo2 = nn.LayerNorm(embed_dim)
        
        # Pairwise representation (simplified from AlphaFold)
        self.pairwise_proj = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim),
            nn.GELU(),
            nn.Linear(embed_dim, embed_dim)
        )
        
        # Gated FFN (inspired by AlphaFold's transition layers)
        self.gate_cc = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Linear(embed_dim * 4, embed_dim),
            nn.Sigmoid()
        )
        self.ffn_cc = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 4, embed_dim)
        )
        
        self.gate_mlo = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Linear(embed_dim * 4, embed_dim),
            nn.Sigmoid()
        )
        self.ffn_mlo = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 4, embed_dim)
        )
        
        self.dropout = nn.Dropout(dropout)
        
        # Store attention weights
        self.cc_attn_weights = None
        self.mlo_attn_weights = None
    
    def forward(self, cc_feat, mlo_feat):
        # Input: [B, embed_dim] -> reshape to [B, 1, embed_dim]
        cc = cc_feat.unsqueeze(1)
        mlo = mlo_feat.unsqueeze(1)
        
        # Bidirectional cross-attention
        cc_attended, cc_attn = self.cc_to_mlo_attn(cc, mlo, mlo)
        mlo_attended, mlo_attn = self.mlo_to_cc_attn(mlo, cc, cc)
        
        # Store attention weights
        self.cc_attn_weights = cc_attn
        self.mlo_attn_weights = mlo_attn
        
        # Residual + norm
        cc = self.norm_cc1(cc + self.dropout(cc_attended))
        mlo = self.norm_mlo1(mlo + self.dropout(mlo_attended))
        
        # Pairwise update (simplified)
        pairwise = self.pairwise_proj(torch.cat([cc, mlo], dim=-1))
        cc = cc + 0.1 * pairwise  # Small residual from pairwise
        mlo = mlo + 0.1 * pairwise
        
        # Gated FFN
        cc_gate = self.gate_cc(cc)
        cc_ffn = self.ffn_cc(cc)
        cc = self.norm_cc2(cc + self.dropout(cc_gate * cc_ffn))
        
        mlo_gate = self.gate_mlo(mlo)
        mlo_ffn = self.ffn_mlo(mlo)
        mlo = self.norm_mlo2(mlo + self.dropout(mlo_gate * mlo_ffn))
        
        # Output: [B, 1, embed_dim] -> [B, embed_dim]
        return cc.squeeze(1), mlo.squeeze(1)


class AlphaBreastV4(nn.Module):
    """
    AlphaBreast V4: Publication-Quality Multi-View Attention Network.
    
    Architecture:
    1. Swin Transformer backbone (pretrained)
    2. Multiple Evoformer-inspired attention blocks
    3. Bidirectional cross-attention (CC ↔ MLO)
    4. Gated FFN with pairwise updates
    
    Inspired by AlphaFold's Evoformer for learning structural relationships.
    """
    
    def __init__(self, num_classes=2, embed_dim=256, num_heads=8, 
                 num_evoformer_blocks=2, pretrained=True):
        super().__init__()
        
        # Shared Swin Transformer encoder
        self.encoder = SwinEncoder(out_features=embed_dim, pretrained=pretrained)
        
        # Stack of Evoformer-inspired blocks
        self.evoformer_blocks = nn.ModuleList([
            EvoformerBlock(embed_dim, num_heads, dropout=0.1)
            for _ in range(num_evoformer_blocks)
        ])
        
        # Final fusion
        self.fusion = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
        
        # For attention visualization
        self.attention_maps = []
    
    def forward(self, cc, mlo, return_attention=False):
        # Encode both views with shared Swin encoder
        cc_feat = self.encoder(cc)  # [B, embed_dim]
        mlo_feat = self.encoder(mlo)  # [B, embed_dim]
        
        # Process through Evoformer blocks
        self.attention_maps = []
        for block in self.evoformer_blocks:
            cc_feat, mlo_feat = block(cc_feat, mlo_feat)
            if return_attention:
                self.attention_maps.append({
                    'cc_attn': block.cc_attn_weights,
                    'mlo_attn': block.mlo_attn_weights
                })
        
        # Fusion
        combined = torch.cat([cc_feat, mlo_feat], dim=1)
        fused = self.fusion(combined)
        
        # Classification
        output = self.classifier(fused)
        
        if return_attention:
            return output, self.attention_maps
        return output


# Test model creation
print("✅ Testing model creation...")
test_model = AlphaBreastV4(num_classes=2, pretrained=True).to(device)
total_params = sum(p.numel() for p in test_model.parameters())
trainable_params = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
del test_model
torch.cuda.empty_cache()

## 5️⃣ Training Functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device, model_type='multi', scaler=None):
    model.train()
    total_loss, correct, total = 0, 0, 0
    
    pbar = tqdm(loader, desc='Train', leave=False)
    for cc, mlo, labels in pbar:
        cc, mlo, labels = cc.to(device), mlo.to(device), labels.to(device)
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast(enabled=scaler is not None):
            if model_type == 'cc_only':
                outputs = model(cc)
            elif model_type == 'mlo_only':
                outputs = model(mlo)
            else:
                outputs = model(cc, mlo)
            loss = criterion(outputs, labels)
        
        if scaler:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        
        total_loss += loss.item()
        _, pred = outputs.max(1)
        total += labels.size(0)
        correct += pred.eq(labels).sum().item()
        
        pbar.set_postfix({'loss': f'{loss.item():.3f}', 'acc': f'{100.*correct/total:.1f}%'})
    
    return total_loss / len(loader), 100. * correct / total


def evaluate(model, loader, device, model_type='multi'):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    
    with torch.no_grad():
        for cc, mlo, labels in loader:
            cc, mlo, labels = cc.to(device), mlo.to(device), labels.to(device)
            
            if model_type == 'cc_only':
                outputs = model(cc)
            elif model_type == 'mlo_only':
                outputs = model(mlo)
            else:
                outputs = model(cc, mlo)
            
            probs = F.softmax(outputs, dim=1)
            _, pred = outputs.max(1)
            
            all_preds.extend(pred.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())
    
    # Metrics
    acc = accuracy_score(all_labels, all_preds) * 100
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except:
        auc = 0.5
    sens = recall_score(all_labels, all_preds, zero_division=0) * 100
    spec_val = precision_score(all_labels, all_preds, zero_division=0) * 100
    f1 = f1_score(all_labels, all_preds, zero_division=0) * 100
    
    cm = confusion_matrix(all_labels, all_preds)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        specificity = tn / (tn + fp) * 100 if (tn + fp) > 0 else 0
    else:
        specificity = 0
    
    return {
        'accuracy': acc,
        'auc': auc,
        'sensitivity': sens,
        'specificity': specificity,
        'f1': f1,
        'predictions': all_preds,
        'labels': all_labels,
        'probs': all_probs
    }

In [ ]:
def train_single_fold(model, train_loader, val_loader, device, model_type='multi',
                      epochs=60, lr=1e-4, patience=15):
    """
    Train model for a single fold.
    Returns best validation metrics.
    """
    criterion = FocalLoss(alpha=0.25, gamma=2.0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
    scaler = torch.cuda.amp.GradScaler()
    
    best_auc = 0
    best_metrics = None
    no_improve = 0
    warmup_epochs = 3
    
    for epoch in range(epochs):
        # Warmup
        if epoch < warmup_epochs:
            for param_group in optimizer.param_groups:
                param_group['lr'] = lr * (epoch + 1) / warmup_epochs
        
        train_loss, train_acc = train_epoch(
            model, train_loader, criterion, optimizer, device, model_type, scaler
        )
        val_metrics = evaluate(model, val_loader, device, model_type)
        
        if epoch >= warmup_epochs:
            scheduler.step()
        
        print(f"  Epoch {epoch+1}/{epochs} | Train: {train_acc:.1f}% | "
              f"Val: {val_metrics['accuracy']:.1f}%, AUC: {val_metrics['auc']:.3f}")
        
        if val_metrics['auc'] > best_auc:
            best_auc = val_metrics['auc']
            best_metrics = val_metrics.copy()
            no_improve = 0
        else:
            no_improve += 1
        
        if no_improve >= patience:
            print(f"  ⏹️ Early stopping at epoch {epoch+1}")
            break
    
    return best_metrics

## 6️⃣ 5-Fold Cross-Validation

In [ ]:
def run_kfold_cv(model_class, model_kwargs, full_dataset, device, model_type='multi',
                 n_folds=5, epochs=60, lr=1e-4, patience=15, batch_size=16):
    """
    Run K-Fold Cross-Validation.
    
    Returns:
        fold_results: List of metrics for each fold
        mean_metrics: Mean ± std for each metric
    """
    print(f"\n{'='*60}")
    print(f"🔄 Starting {n_folds}-Fold Cross-Validation")
    print(f"{'='*60}")
    
    # Get labels for stratified split
    labels = full_dataset.labels
    
    # Stratified K-Fold
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    
    fold_results = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
        print(f"\n📁 Fold {fold+1}/{n_folds}")
        print(f"   Train: {len(train_idx)}, Val: {len(val_idx)}")
        
        # Create fold datasets
        train_subset = Subset(full_dataset, train_idx)
        val_subset = Subset(full_dataset, val_idx)
        
        # Apply transforms
        class TransformedSubset(Dataset):
            def __init__(self, subset, transform):
                self.subset = subset
                self.transform = transform
            
            def __len__(self):
                return len(self.subset)
            
            def __getitem__(self, idx):
                sample = self.subset.dataset.valid_samples[self.subset.indices[idx]]
                cc_img = Image.open(sample['cc_path']).convert('L')
                mlo_img = Image.open(sample['mlo_path']).convert('L')
                
                if self.transform:
                    seed = np.random.randint(2147483647)
                    torch.manual_seed(seed)
                    cc_img = self.transform(cc_img)
                    torch.manual_seed(seed)
                    mlo_img = self.transform(mlo_img)
                
                return cc_img, mlo_img, torch.tensor(sample['label'], dtype=torch.long)
        
        train_data = TransformedSubset(train_subset, train_transform)
        val_data = TransformedSubset(val_subset, test_transform)
        
        train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, 
                                  num_workers=2, pin_memory=True)
        val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False,
                                num_workers=2, pin_memory=True)
        
        # Create fresh model for this fold
        model = model_class(**model_kwargs).to(device)
        
        # Train
        best_metrics = train_single_fold(
            model, train_loader, val_loader, device, model_type,
            epochs=epochs, lr=lr, patience=patience
        )
        
        fold_results.append({
            'fold': fold + 1,
            'accuracy': best_metrics['accuracy'],
            'auc': best_metrics['auc'],
            'sensitivity': best_metrics['sensitivity'],
            'specificity': best_metrics['specificity'],
            'f1': best_metrics['f1']
        })
        
        print(f"   ✅ Fold {fold+1} Best: Acc={best_metrics['accuracy']:.1f}%, AUC={best_metrics['auc']:.3f}")
        
        # Clean up
        del model
        torch.cuda.empty_cache()
    
    # Calculate mean ± std
    results_df = pd.DataFrame(fold_results)
    
    mean_metrics = {
        'accuracy_mean': results_df['accuracy'].mean(),
        'accuracy_std': results_df['accuracy'].std(),
        'auc_mean': results_df['auc'].mean(),
        'auc_std': results_df['auc'].std(),
        'sensitivity_mean': results_df['sensitivity'].mean(),
        'sensitivity_std': results_df['sensitivity'].std(),
        'specificity_mean': results_df['specificity'].mean(),
        'specificity_std': results_df['specificity'].std(),
        'f1_mean': results_df['f1'].mean(),
        'f1_std': results_df['f1'].std()
    }
    
    print(f"\n{'='*60}")
    print(f"📊 {n_folds}-Fold CV Results:")
    print(f"   Accuracy:    {mean_metrics['accuracy_mean']:.1f}% ± {mean_metrics['accuracy_std']:.1f}%")
    print(f"   AUC-ROC:     {mean_metrics['auc_mean']:.3f} ± {mean_metrics['auc_std']:.3f}")
    print(f"   Sensitivity: {mean_metrics['sensitivity_mean']:.1f}% ± {mean_metrics['sensitivity_std']:.1f}%")
    print(f"   Specificity: {mean_metrics['specificity_mean']:.1f}% ± {mean_metrics['specificity_std']:.1f}%")
    print(f"{'='*60}")
    
    return fold_results, mean_metrics

## 7️⃣ Run All Experiments

In [ ]:
# Store all results
all_cv_results = {}
all_fold_results = {}

N_FOLDS = 5
EPOCHS = 60
PATIENCE = 15
BATCH_SIZE = 16
LR = 1e-4

In [ ]:
# Model 1: CC Only (Swin)
print("\n" + "="*70)
print("🏋️ Model 1: Single-View Swin Transformer (CC Only)")
print("="*70)

fold_results, mean_metrics = run_kfold_cv(
    model_class=SingleViewSwin,
    model_kwargs={'num_classes': 2, 'pretrained': True},
    full_dataset=full_train_dataset,
    device=device,
    model_type='cc_only',
    n_folds=N_FOLDS,
    epochs=EPOCHS,
    lr=LR,
    patience=PATIENCE,
    batch_size=BATCH_SIZE
)

all_cv_results['CC Only'] = mean_metrics
all_fold_results['CC Only'] = fold_results

In [ ]:
# Model 2: MLO Only (Swin)
print("\n" + "="*70)
print("🏋️ Model 2: Single-View Swin Transformer (MLO Only)")
print("="*70)

fold_results, mean_metrics = run_kfold_cv(
    model_class=SingleViewSwin,
    model_kwargs={'num_classes': 2, 'pretrained': True},
    full_dataset=full_train_dataset,
    device=device,
    model_type='mlo_only',
    n_folds=N_FOLDS,
    epochs=EPOCHS,
    lr=LR,
    patience=PATIENCE,
    batch_size=BATCH_SIZE
)

all_cv_results['MLO Only'] = mean_metrics
all_fold_results['MLO Only'] = fold_results

In [ ]:
# Model 3: Late Fusion (Swin)
print("\n" + "="*70)
print("🏋️ Model 3: Late Fusion Swin Transformer")
print("="*70)

fold_results, mean_metrics = run_kfold_cv(
    model_class=LateFusionSwin,
    model_kwargs={'num_classes': 2, 'pretrained': True},
    full_dataset=full_train_dataset,
    device=device,
    model_type='multi',
    n_folds=N_FOLDS,
    epochs=EPOCHS,
    lr=LR,
    patience=PATIENCE,
    batch_size=BATCH_SIZE
)

all_cv_results['Late Fusion'] = mean_metrics
all_fold_results['Late Fusion'] = fold_results

In [ ]:
# Model 4: AlphaBreast V4
print("\n" + "="*70)
print("🏋️ Model 4: AlphaBreast V4 (Swin + Evoformer-Inspired Attention)")
print("="*70)

fold_results, mean_metrics = run_kfold_cv(
    model_class=AlphaBreastV4,
    model_kwargs={
        'num_classes': 2,
        'embed_dim': 256,
        'num_heads': 8,
        'num_evoformer_blocks': 2,
        'pretrained': True
    },
    full_dataset=full_train_dataset,
    device=device,
    model_type='multi',
    n_folds=N_FOLDS,
    epochs=EPOCHS,
    lr=LR,
    patience=PATIENCE,
    batch_size=BATCH_SIZE
)

all_cv_results['AlphaBreast V4'] = mean_metrics
all_fold_results['AlphaBreast V4'] = fold_results

## 8️⃣ Final Results

In [ ]:
print("\n" + "="*80)
print("📊 FINAL RESULTS - AlphaBreast V4 with 5-Fold Cross-Validation")
print("="*80)

# Create summary table
summary_data = []
for model_name, metrics in all_cv_results.items():
    summary_data.append({
        'Model': model_name,
        'Accuracy': f"{metrics['accuracy_mean']:.1f}% ± {metrics['accuracy_std']:.1f}%",
        'AUC-ROC': f"{metrics['auc_mean']:.3f} ± {metrics['auc_std']:.3f}",
        'Sensitivity': f"{metrics['sensitivity_mean']:.1f}% ± {metrics['sensitivity_std']:.1f}%",
        'Specificity': f"{metrics['specificity_mean']:.1f}% ± {metrics['specificity_std']:.1f}%",
        'F1 Score': f"{metrics['f1_mean']:.1f}% ± {metrics['f1_std']:.1f}%"
    })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

# Save to CSV
summary_df.to_csv(os.path.join(OUTPUT_DIR, 'v4_5fold_results.csv'), index=False)
print(f"\n💾 Results saved to {OUTPUT_DIR}/v4_5fold_results.csv")

In [ ]:
# Improvement analysis
print("\n" + "="*60)
print("📈 IMPROVEMENT ANALYSIS")
print("="*60)

alpha_acc = all_cv_results['AlphaBreast V4']['accuracy_mean']
alpha_auc = all_cv_results['AlphaBreast V4']['auc_mean']

cc_acc = all_cv_results['CC Only']['accuracy_mean']
mlo_acc = all_cv_results['MLO Only']['accuracy_mean']
late_acc = all_cv_results['Late Fusion']['accuracy_mean']

cc_auc = all_cv_results['CC Only']['auc_mean']
mlo_auc = all_cv_results['MLO Only']['auc_mean']
late_auc = all_cv_results['Late Fusion']['auc_mean']

avg_single_acc = (cc_acc + mlo_acc) / 2
avg_single_auc = (cc_auc + mlo_auc) / 2

print(f"\n📊 Accuracy Improvements:")
print(f"   AlphaBreast V4 vs CC Only:      {alpha_acc - cc_acc:+.1f}%")
print(f"   AlphaBreast V4 vs MLO Only:     {alpha_acc - mlo_acc:+.1f}%")
print(f"   AlphaBreast V4 vs Single-View Avg: {alpha_acc - avg_single_acc:+.1f}%")
print(f"   AlphaBreast V4 vs Late Fusion:  {alpha_acc - late_acc:+.1f}%")

print(f"\n📊 AUC Improvements:")
print(f"   AlphaBreast V4 vs CC Only:      {alpha_auc - cc_auc:+.3f}")
print(f"   AlphaBreast V4 vs MLO Only:     {alpha_auc - mlo_auc:+.3f}")
print(f"   AlphaBreast V4 vs Single-View Avg: {alpha_auc - avg_single_auc:+.3f}")
print(f"   AlphaBreast V4 vs Late Fusion:  {alpha_auc - late_auc:+.3f}")

In [ ]:
# Visualization: Box plots for each fold
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Prepare data for box plots
models = ['CC Only', 'MLO Only', 'Late Fusion', 'AlphaBreast V4']
colors = ['#3498db', '#3498db', '#e74c3c', '#2ecc71']

# Accuracy box plot
acc_data = [[r['accuracy'] for r in all_fold_results[m]] for m in models]
bp1 = axes[0].boxplot(acc_data, labels=models, patch_artist=True)
for patch, color in zip(bp1['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0].set_ylabel('Accuracy (%)', fontsize=12)
axes[0].set_title('5-Fold CV: Accuracy Distribution', fontsize=14, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# AUC box plot
auc_data = [[r['auc'] for r in all_fold_results[m]] for m in models]
bp2 = axes[1].boxplot(auc_data, labels=models, patch_artist=True)
for patch, color in zip(bp2['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_ylabel('AUC-ROC', fontsize=12)
axes[1].set_title('5-Fold CV: AUC Distribution', fontsize=14, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'v4_5fold_boxplots.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"💾 Box plots saved to {OUTPUT_DIR}/v4_5fold_boxplots.png")

In [ ]:
# Bar chart with error bars
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

models = ['CC Only', 'MLO Only', 'Late Fusion', 'AlphaBreast V4']
colors = ['#3498db', '#3498db', '#e74c3c', '#2ecc71']
x = np.arange(len(models))

# Accuracy
acc_means = [all_cv_results[m]['accuracy_mean'] for m in models]
acc_stds = [all_cv_results[m]['accuracy_std'] for m in models]
bars1 = axes[0].bar(x, acc_means, yerr=acc_stds, capsize=5, color=colors, alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(models, rotation=15)
axes[0].set_ylabel('Accuracy (%)', fontsize=12)
axes[0].set_title('Model Accuracy (Mean ± Std)', fontsize=14, fontweight='bold')
axes[0].set_ylim([0, 100])
for i, (m, s) in enumerate(zip(acc_means, acc_stds)):
    axes[0].text(i, m + s + 2, f'{m:.1f}%', ha='center', fontweight='bold')

# AUC
auc_means = [all_cv_results[m]['auc_mean'] for m in models]
auc_stds = [all_cv_results[m]['auc_std'] for m in models]
bars2 = axes[1].bar(x, auc_means, yerr=auc_stds, capsize=5, color=colors, alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(models, rotation=15)
axes[1].set_ylabel('AUC-ROC', fontsize=12)
axes[1].set_title('Model AUC (Mean ± Std)', fontsize=14, fontweight='bold')
axes[1].set_ylim([0, 1])
for i, (m, s) in enumerate(zip(auc_means, auc_stds)):
    axes[1].text(i, m + s + 0.03, f'{m:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'v4_5fold_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"💾 Comparison chart saved to {OUTPUT_DIR}/v4_5fold_comparison.png")

In [ ]:
# Fold-by-fold results table
print("\n" + "="*80)
print("📋 DETAILED FOLD-BY-FOLD RESULTS")
print("="*80)

for model_name in models:
    print(f"\n{model_name}:")
    fold_df = pd.DataFrame(all_fold_results[model_name])
    print(fold_df.to_string(index=False))

# Save detailed results
with open(os.path.join(OUTPUT_DIR, 'v4_detailed_results.json'), 'w') as f:
    json.dump({
        'cv_results': {k: {kk: float(vv) if isinstance(vv, (np.floating, float)) else vv 
                          for kk, vv in v.items()} 
                      for k, v in all_cv_results.items()},
        'fold_results': all_fold_results
    }, f, indent=2)
print(f"\n💾 Detailed results saved to {OUTPUT_DIR}/v4_detailed_results.json")

## 9️⃣ Copy for Final Report

In [ ]:
print("\n" + "="*80)
print("📝 COPY TO FINAL REPORT - Table 7: Performance with 5-Fold CV")
print("="*80)

print("\n| Model | Accuracy | AUC-ROC | Sensitivity | Specificity |")
print("|-------|----------|---------|-------------|-------------|")

for model in models:
    m = all_cv_results[model]
    print(f"| {model} | {m['accuracy_mean']:.1f}% ± {m['accuracy_std']:.1f}% | "
          f"{m['auc_mean']:.3f} ± {m['auc_std']:.3f} | "
          f"{m['sensitivity_mean']:.1f}% ± {m['sensitivity_std']:.1f}% | "
          f"{m['specificity_mean']:.1f}% ± {m['specificity_std']:.1f}% |")

In [ ]:
# Version comparison table
print("\n" + "="*80)
print("📝 COPY TO FINAL REPORT - Table 8: Version Evolution")
print("="*80)

print("\n| Version | Architecture | Data | Accuracy | AUC-ROC |")
print("|---------|--------------|------|----------|---------|")
print("| V1 | 3-layer CNN, Single Attention | 509 Mass pairs | 51.0% | 0.614 |")
print("| V2 | ResNet18, Bidirectional Attention | 509 Mass pairs | 60.3% | 0.72 |")
print("| V3 | ResNet18, Bidirectional Attention | ~1000 Mass+Calc pairs | 68.0% | 0.74 |")
print(f"| **V4** | **Swin + Evoformer, 5-Fold CV** | **~1000 Mass+Calc pairs** | "
      f"**{all_cv_results['AlphaBreast V4']['accuracy_mean']:.1f}% ± {all_cv_results['AlphaBreast V4']['accuracy_std']:.1f}%** | "
      f"**{all_cv_results['AlphaBreast V4']['auc_mean']:.3f} ± {all_cv_results['AlphaBreast V4']['auc_std']:.3f}** |")

## 🎉 Training Complete!

**Files Saved:**
- `v4_5fold_results.csv` - Summary results
- `v4_detailed_results.json` - All fold details
- `v4_5fold_boxplots.png` - Box plot visualization
- `v4_5fold_comparison.png` - Bar chart with error bars

**For Your Report:**
1. Copy Table 7 (5-Fold CV results)
2. Copy Table 8 (Version evolution V1→V4)
3. Download figures for Chapter 5

**Key Achievements:**
- ✅ Swin Transformer backbone (State-of-the-art)
- ✅ Evoformer-inspired attention (AlphaFold-style)
- ✅ 5-Fold Cross-Validation (Publication standard)
- ✅ Combined Mass + Calcification data